# 04. Multi-Query Retrieval
## 1 Question → Multiple Queries

### 이번 실습의 위치

```text
01 Naive RAG
    ↓
02 Document Processing
    ↓
03 Query Rewriting
   1 Question → 1 Better Query
    ↓
04 Multi Query
   1 Question → Multiple Queries
    ↓
05 Retrieval Engineering
```

이번 실습의 핵심은 하나이다.

> **하나의 질문을 여러 검색 표현으로 바꾸면, 한 번의 검색에서 놓친 관련 문서를 더 찾을 수 있다.**

### 학습 목표

1. Query Rewriting과 Multi Query의 차이를 설명한다.
2. 하나의 질문에서 여러 검색 질문을 만든다.
3. 각 질문으로 검색한 결과를 하나로 합친다.
4. 같은 문서는 중복 제거한다.
5. Single Query보다 검색 범위가 넓어질 수 있음을 확인한다.


# 1. 환경 설정

02번에서 만든 ChromaDB를 그대로 사용한다.

이번 실습에서는 검색기 자체를 바꾸지 않는다.

```text
Single Query
질문 1개 → Dense Search 1회

Multi Query
질문 1개 → 검색 질문 여러 개 → Dense Search 여러 회
```


In [3]:
# Python이 상위 폴더의 common_config.py를 찾을 수 있도록 경로를 추가한다.
import sys
from pathlib import Path

# ChromaDB에 저장된 벡터 데이터를 불러오기 위한 클래스
from langchain_chroma import Chroma

sys.path.append("..")

# 공통 설정 파일에서 임베딩 모델과 LLM 연결 함수를 가져온다.
from common_config import embedding_model, llm_connect


# 02번 실습에서 만든 ChromaDB의 위치와 컬렉션 이름
DB_PATH = Path("chroma_db/ai_trend_report_naive_markdown")
COLLECTION_NAME = "ai_trend_report_2026_naive_markdown"

# 02번 실습을 먼저 실행하지 않았다면 검색할 데이터가 없으므로 중단한다.
if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_ai_trend.ipynb를 먼저 실행하세요."
    )

# Query를 vector로 변환할 임베딩 모델 생성
embeddings = embedding_model()

# Multi Query를 생성할 LLM 연결
# temperature=0으로 설정하여 가능한 한 일관된 질문을 생성하도록 한다.
llm = llm_connect(
    temperature=0,
    max_tokens=300,
)

# 기존 ChromaDB를 새로 만들지 않고 그대로 연결한다.
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

# 정상적으로 DB가 연결되었는지 저장된 chunk 수로 확인한다.
print("저장된 chunk 수:", vector_store._collection.count())


저장된 chunk 수: 73


# 2. 먼저 Single Query로 검색

먼저 질문 하나로 검색한다.

```text
사용자 질문
    ↓
Vector Search
    ↓
Top-3
```

이 결과를 기준으로 나중에 Multi Query 결과와 비교한다.


In [4]:
# Multi Query와 비교하기 위한 원본 질문
question = "2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?"

# 원본 질문 하나만 사용하여 Dense Retrieval을 수행한다.
# k=3이므로 의미적으로 가장 가까운 문서 3개를 가져온다.
single_docs = vector_store.similarity_search(
    question,
    k=3,
)

print("[Single Query]")

# 검색 결과의 순위와 문서 위치를 확인한다.
for rank, doc in enumerate(single_docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


[Single Query]
1. chunk=9 | pages=None
2. chunk=59 | pages=None
3. chunk=55 | pages=None


# 3. Query Rewriting과 Multi Query의 차이

둘은 비슷해 보이지만 목적이 다르다.

| 방법 | 결과 | 목적 |
|---|---|---|
| Query Rewriting | 더 좋은 질문 1개 | 질문을 명확하게 만든다 |
| Multi Query | 검색 질문 여러 개 | 검색 범위를 넓힌다 |

```text
Query Rewriting
Q → Q'

Multi Query
Q → Q1
  → Q2
  → Q3
```

Multi Query는 **다른 주제를 만드는 것이 아니다.**

원래 질문의 의미를 유지하면서 표현만 다르게 만든다.


# 4. LLM으로 검색 질문 3개 만들기

예를 들어 다음 질문이 있다고 하자.

```text
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
```

Multi Query는 같은 정보 요구를 다음처럼 여러 표현으로 바꾼다.

```text
Q1. 2026년 AI 기술의 주요 발전 방향은 무엇인가?
Q2. AI 기술은 어떤 기능과 형태로 고도화될 것으로 예상되는가?
Q3. 앞으로 AI 기술에서 중요해질 변화는 무엇인가?
```

핵심은 **원래 질문의 범위를 벗어나지 않는 것**이다.


In [8]:
# LLM에게 원본 질문을 서로 다른 표현의 검색 질문 3개로 바꾸도록 지시한다.
# 중요한 점은 "새로운 주제를 만드는 것"이 아니라
# "같은 정보 요구를 다른 표현으로 검색하는 것"이다.
prompt = f"""
다음 질문을 RAG 검색에 사용할 검색 질문 3개로 바꿔주세요.

규칙:
- 원래 질문의 의미를 유지하세요.
- 새로운 주제를 추가하지 마세요.
- 서로 다른 표현으로 작성하세요.
- 한 줄에 질문 하나만 작성하세요.
- 번호는 붙이지 마세요.

원본 질문:
{question}
"""

# LLM에게 프롬프트를 전달한다.
response = llm.invoke(prompt)

response
# # LLM의 문자열 응답을 줄 단위로 나눈다.
# # 빈 줄은 제거하고 앞의 3개 질문만 사용한다.
# generated_queries = [
#     line.strip()
#     for line in response.content.splitlines()
#     if line.strip()
# ][:3]

# print("[생성된 Multi Queries]")

# # 실제 생성된 검색 질문을 확인한다.
# for i, query in enumerate(generated_queries, start=1):
#     print(f"{i}. {query}")


AIMessage(content='2026년 AI 기술은 어떤 방향으로 발전할 것으로 예상되나요?\n2026년에 인공지능 기술은 어떤 방향으로 발전할 것으로 보이나요?\n2026년 AI 기술 발전의 주요 전망은 무엇인가요?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 96, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-mini', 'system_fingerprint': None, 'id': 'chatcmpl-EVt90A7NKYzmDjKiBb1F2LysQXEcR', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a10ff1-870a-71f1-aab0-a4dcaab7b104-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 96, 'output_tokens': 51, 'total_tokens': 147, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_

In [9]:

# LLM의 문자열 응답을 줄 단위로 나눈다.
# 빈 줄은 제거하고 앞의 3개 질문만 사용한다.
generated_queries = [
    line.strip()
    for line in response.content.splitlines()
    if line.strip()
][:3]

print("[생성된 Multi Queries]")

# 실제 생성된 검색 질문을 확인한다.
for i, query in enumerate(generated_queries, start=1):
    print(f"{i}. {query}")


[생성된 Multi Queries]
1. 2026년 AI 기술은 어떤 방향으로 발전할 것으로 예상되나요?
2. 2026년에 인공지능 기술은 어떤 방향으로 발전할 것으로 보이나요?
3. 2026년 AI 기술 발전의 주요 전망은 무엇인가요?


# 5. 여러 Query로 검색하고 결과 합치기

이제 원본 질문과 생성된 질문을 모두 검색한다.

```text
원본 질문 ─────→ Top-3
Query 1 ──────→ Top-3
Query 2 ──────→ Top-3
Query 3 ──────→ Top-3
                  ↓
              결과 합치기
                  ↓
              중복 제거
```

같은 chunk가 여러 번 검색될 수 있으므로 `chunk_id`를 이용해 한 번만 남긴다.


In [6]:
# 원본 질문도 버리지 않고 함께 사용한다.
# 총 검색 질문 = 원본 질문 1개 + LLM이 만든 질문 3개
search_queries = [question] + generated_queries

# 최종적으로 사용할 문서를 저장할 리스트
multi_docs = []

# 같은 chunk가 여러 Query에서 반복 검색될 수 있으므로
# 이미 저장한 chunk_id를 기억하기 위한 set
seen_chunk_ids = set()

# 각 Query를 하나씩 검색한다.
for query in search_queries:
    print("\n[검색 질문]")
    print(query)

    # 현재 Query로 상위 3개 문서를 검색한다.
    docs = vector_store.similarity_search(
        query,
        k=3,
    )

    # 검색된 문서를 하나씩 확인한다.
    for doc in docs:
        chunk_id = doc.metadata.get("chunk_id")

        print(
            f"  chunk={chunk_id} "
            f"| pages={doc.metadata.get('pages')}"
        )

        # 동일한 chunk가 아직 저장되지 않았다면 추가한다.
        # set을 사용하면 중복 여부를 빠르게 확인할 수 있다.
        if chunk_id not in seen_chunk_ids:
            seen_chunk_ids.add(chunk_id)
            multi_docs.append(doc)



[검색 질문]
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
  chunk=9 | pages=None
  chunk=59 | pages=None
  chunk=55 | pages=None

[검색 질문]
2026년 AI 기술은 어떤 방향으로 발전할 것으로 예상되나요?
  chunk=9 | pages=None
  chunk=59 | pages=None
  chunk=8 | pages=None

[검색 질문]
2026년에 AI 기술은 어떤 추세로 발전할 전망인가요?
  chunk=9 | pages=None
  chunk=55 | pages=None
  chunk=58 | pages=None

[검색 질문]
2026년의 AI 기술 발전 방향은 어떻게 전망되나요?
  chunk=9 | pages=None
  chunk=55 | pages=None
  chunk=8 | pages=None


# 6. Single Query vs Multi Query 비교

Multi Query의 목적은 단순히 문서 수를 늘리는 것이 아니다.

확인할 것은 다음 두 가지면 충분하다.

1. Single Query에서 찾지 못한 관련 문서가 추가되었는가?
2. 불필요한 문서도 같이 늘어나지는 않았는가?

즉, Multi Query는 일반적으로 **Recall을 높이기 위한 검색 전략**이다.


In [7]:
# Single Query 검색 결과의 chunk_id만 집합으로 만든다.
single_ids = {
    doc.metadata.get("chunk_id")
    for doc in single_docs
}

# Multi Query 검색 결과의 chunk_id만 집합으로 만든다.
multi_ids = {
    doc.metadata.get("chunk_id")
    for doc in multi_docs
}

# Multi Query에서 새롭게 추가된 문서를 확인한다.
# 집합의 차집합을 사용하면 Single Query에는 없고
# Multi Query에만 존재하는 chunk를 찾을 수 있다.
added_ids = multi_ids - single_ids

print("Single Query 문서 수:", len(single_ids))
print("Multi Query 문서 수 :", len(multi_ids))

print("\nMulti Query에서 새롭게 추가된 chunk:")
print(sorted(added_ids))


Single Query 문서 수: 3
Multi Query 문서 수 : 5

Multi Query에서 새롭게 추가된 chunk:
[8, 58]


In [12]:
# ==========================================
# Single Query vs Multi Query 비교 테스트
# ==========================================

# 1. Document 리스트를 LLM용 Context 문자열로 변환
def docs_to_context(docs):
    context_list = []

    for doc in docs:
        chunk_id = doc.metadata.get("chunk_id")
        pages = doc.metadata.get("pages")
        content = doc.page_content

        context_list.append(
            f"[chunk={chunk_id}, pages={pages}]\n{content}"
        )

    return "\n\n".join(context_list)


# 2. 검색 결과 개수 및 chunk_id 확인
print("=" * 60)
print("[검색 결과 비교]")

print("Single Query Chunk 수:", len(single_docs))
print(
    "Single Query Chunk IDs:",
    [doc.metadata.get("chunk_id") for doc in single_docs]
)

print()

print("Multi Query Chunk 수:", len(multi_docs))
print(
    "Multi Query Chunk IDs:",
    [doc.metadata.get("chunk_id") for doc in multi_docs]
)


# 3. Multi Query에서 새롭게 추가된 Chunk 확인
single_ids = {
    doc.metadata.get("chunk_id")
    for doc in single_docs
}

multi_ids = {
    doc.metadata.get("chunk_id")
    for doc in multi_docs
}

added_ids = multi_ids - single_ids

print()
print("Multi Query에서 추가로 검색된 Chunk:")
print(sorted(added_ids))


# 4. LLM에 넣을 Context 생성
single_context = docs_to_context(single_docs)
multi_context = docs_to_context(multi_docs)


# 5. Single Query 답변용 Prompt
single_prompt = f"""
아래 검색 문서만 참고하여 질문에 답하세요.
문서에 없는 내용은 임의로 추가하지 마세요.

[질문]
{question}

[검색 문서]
{single_context}

[답변]
"""


# 6. Multi Query 답변용 Prompt
multi_prompt = f"""
아래 검색 문서만 참고하여 질문에 답하세요.
문서에 없는 내용은 임의로 추가하지 마세요.

[질문]
{question}

[검색 문서]
{multi_context}

[답변]
"""


# 7. 동일한 LLM으로 답변 생성
single_response = llm.invoke(single_prompt)
multi_response = llm.invoke(multi_prompt)


# 8. 최종 결과 출력
print("\n" + "=" * 60)
print("[원본 질문]")
print(question)


print("\n" + "=" * 60)
print("[Single Query에서 실제 참고한 Chunk]")

for rank, doc in enumerate(single_docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


print("\n[Single Query 답변]")
print(single_response.content)


print("\n" + "=" * 60)
print("[Multi Query에서 실제 참고한 Chunk]")

for rank, doc in enumerate(multi_docs, start=1):
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


print("\n[Multi Query 답변]")
print(multi_response.content)


print("\n" + "=" * 60)
print("[비교 요약]")

print(
    f"Single Query는 {len(single_docs)}개의 Chunk를 참고했습니다."
)

print(
    f"Multi Query는 {len(multi_docs)}개의 고유 Chunk를 참고했습니다."
)

print(
    f"Multi Query에서 새롭게 추가된 Chunk는 "
    f"{len(added_ids)}개입니다: {sorted(added_ids)}"
)

[검색 결과 비교]
Single Query Chunk 수: 3
Single Query Chunk IDs: [9, 59, 55]

Multi Query Chunk 수: 5
Multi Query Chunk IDs: [9, 59, 55, 8, 58]

Multi Query에서 추가로 검색된 Chunk:
[8, 58]

[원본 질문]
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?

[Single Query에서 실제 참고한 Chunk]
1. chunk=9 | pages=None
2. chunk=59 | pages=None
3. chunk=55 | pages=None

[Single Query 답변]
2026년 AI 기술은 **지능 구조의 고도화**와 **데이터 한계 보완**을 중심으로 발전할 것으로 전망됩니다. 구체적으로는:

- **합성데이터, 추론형 AI, 멀티모달 기술**이 주요 경쟁 축이 될 가능성이 큼
- 이를 통해 **학습 효율 향상**, **복합 정보 처리**, **설명 가능성 강화** 같은 모델 내부 구조의 질적 개선이 이어질 것으로 예상됨
- **입력 유형과 데이터 범위가 확장**되면서 AI의 활용 가능성이 넓어지고, **복잡한 상황에서의 판단 능력**도 강화될 것으로 전망됨
- **고품질 데이터 생성 + 멀티모달 + 고급 추론**이 결합되어 **상황 이해와 문제 해결 능력**이 향상되고, 서비스·산업 전반의 활용도도 확대될 것으로 보임
- 동시에 기술 고도화로 인해 **안전성·정합성 검증**의 중요성이 커지고, **신뢰성 평가와 표준화**가 확산의 기반 인프라가 될 것으로 전망됩니다.

[Multi Query에서 실제 참고한 Chunk]
1. chunk=9 | pages=None
2. chunk=59 | pages=None
3. chunk=55 | pages=None
4. chunk=8 | pages=None
5. chunk=58 | pages=None

[Multi Query 답변]
2026년 AI 기술은 **지능 구조의 고도화**

In [11]:
print("생성된 검색 Query 수:", len(search_queries))

for i, query in enumerate(search_queries, start=1):
    print(f"{i}. {query}")

print("\n최종 Multi Chunk 수:", len(multi_docs))

for doc in multi_docs:
    print(doc.metadata.get("chunk_id"))

생성된 검색 Query 수: 4
1. 2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
2. 2026년 AI 기술은 어떤 방향으로 발전할 것으로 예상되나요?
3. 2026년에 AI 기술은 어떤 추세로 발전할 전망인가요?
4. 2026년의 AI 기술 발전 방향은 어떻게 전망되나요?

최종 Multi Chunk 수: 5
9
59
55
8
58


# 7. 핵심 정리

## 오늘 기억할 것

```text
Single Query
질문 1개
   ↓
검색 1회


Multi Query
질문 1개
   ↓
검색 질문 여러 개
   ↓
각각 검색
   ↓
결과 합치기
   ↓
중복 제거
```

### 장점

- 한 번의 검색에서 놓친 관련 문서를 추가로 찾을 수 있다.
- 표현이 다른 문서를 찾을 가능성이 높아진다.
- Recall을 높이는 데 도움이 된다.

### 한계

- 검색 횟수가 증가한다.
- LLM으로 Query를 만드는 비용이 추가된다.
- 검색 범위가 넓어지면서 noise가 증가할 수 있다.
- 생성된 Query가 원래 질문에서 벗어날 수 있다.

> Multi Query는 **더 많은 문서를 가져오는 기술**이 아니라,  
> **같은 정보 요구를 여러 표현으로 검색하여 놓치는 문서를 줄이는 기술**이다.


# 8. 다음 실습으로 연결

현재까지의 흐름은 다음과 같다.

```text
03 Query Rewriting
1 Question
    ↓
1 Better Query

04 Multi Query
1 Question
    ↓
Multiple Queries
    ↓
더 넓은 후보 문서

05 Retrieval Engineering
Dense
  +
BM25
  ↓
Hybrid + RRF
```

04번에서는 **검색 질문을 다양하게 만드는 방법**에 집중한다.

05번에서는 검색 질문이 아니라 **검색기 자체를 개선하는 방법**으로 넘어간다.
